# Non-Negative Baselines

This notebook shows the current conservative non-negativity behavior for `DebiasConvex` and `MCNNMPanelSolver`.

- `DebiasConvex(method_non_neg=...)` is treated as experimental point estimation. Standard errors are not returned as inference-valid.
- `MCNNMPanelSolver(baseline_projection="clip_nonnegative")` keeps raw outputs and adds projected companion outputs.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from causaltensor.cauest.DebiasConvex import DCPanelSolver
from causaltensor.cauest.MCNNM import MCNNMPanelSolver

## DebiasConvex

The non-negative modes are available, but they are marked as experimental because they change the low-rank geometry used by the debiasing and standard-error formulas.

In [2]:
M0 = np.outer(np.linspace(1.0, 2.0, 8), np.linspace(0.5, 1.5, 8))
Z = np.zeros_like(M0)
Z[4:, 4:] = 1
O = M0 + 0.25 * Z

solver = DCPanelSolver(Z=Z, O=O)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    res_dc = solver.fit(suggest_r=1, method="non-convex", method_non_neg="svd")

print("warnings:", len(caught))
print("tau:", round(float(res_dc.tau), 4))
print("std:", res_dc.std)
print("inference_valid:", res_dc.inference_valid)
print("diagnostics:")
for key in ["method_non_neg", "std_available", "rank", "stable_rank", "projected_design_condition_number"]:
    print(f"  {key}: {res_dc.diagnostics[key]}")

warnings: 1
tau: 0.25
std: None
inference_valid: False
diagnostics:
  method_non_neg: svd
  std_available: False
  rank: 1
  stable_rank: 1.0
  projected_design_condition_number: 1.0


## MCNNMPanelSolver

For MCNNM, the low-rank component `M` is not the right non-negativity target when fixed effects are present. The optional projection is applied to the final baseline as post-processing, while raw outputs remain available.

In [3]:
rng = np.random.default_rng(1)
shape = (12, 12)
rank = 2
U = rng.normal(size=(shape[0], rank))
V = rng.normal(size=(shape[1], rank))
baseline_true = U @ V.T
baseline_true = baseline_true - np.min(baseline_true) + 0.05
baseline_true = np.maximum(baseline_true - np.quantile(baseline_true, 0.35), 0)

Z = np.zeros(shape)
Z[shape[0] // 2:, shape[1] // 2:] = 1
tau_true = 0.2
O = baseline_true + tau_true * Z + rng.normal(scale=0.2, size=shape)

solver = MCNNMPanelSolver(Z=Z)
res_mc = solver.fit(O=O, l=1.0, max_iter=500, baseline_projection="clip_nonnegative")

print("raw baseline min:", round(float(np.min(res_mc.baseline)), 4))
print("projected baseline min:", round(float(np.min(res_mc.baseline_projected)), 4))
print("raw tau:", round(float(res_mc.tau), 4))
print("projected tau:", round(float(res_mc.tau_projected), 4))
print("projection diagnostics:")
for key in ["clipped_fraction", "clipped_mass", "max_clipped_magnitude", "tau_shift"]:
    print(f"  {key}: {res_mc.projection_diagnostics[key]:.4f}")

raw baseline min: -0.2825
projected baseline min: 0.0
raw tau: 0.274
projected tau: 0.2375
projection diagnostics:
  clipped_fraction: 0.1806
  clipped_mass: 3.4207
  max_clipped_magnitude: 0.2825
  tau_shift: -0.0366
